In [5]:
import pandas as pd

In [7]:
bank = pd.read_csv('bank-additional-full.csv.zip',sep = ';')
bank.drop(columns=['default'],inplace=True)

In [8]:
bank['previously_contacted'] = (bank['pdays'] != 999).astype(int)

In [9]:
bank = bank[bank['education'] != 'unknown']
bank = bank[bank['loan'] != 'unknown']
bank.shape

(38507, 21)

In [10]:
(bank['y'].value_counts()/bank.shape[0])*100

,count
y,
no,88.866959
yes,11.133041


In [11]:
x = bank.drop(columns=['y'])
y = bank['y']

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

Y_scaled = le.fit_transform(y)

In [13]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(x,Y_scaled, test_size = .2 , random_state = 42 )

In [19]:
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OrdinalEncoder

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import roc_auc_score

from sklearn.metrics import classification_report

from xgboost import XGBClassifier

In [14]:
education_order = [
    'illiterate',
    'basic.4y',
    'basic.6y',
    'basic.9y',
    'high.school',
    'professional.course',
    'university.degree'
]


transform = ColumnTransformer(transformers=[
    ('ore',OrdinalEncoder(categories=[education_order]),['education']),
    ('ohe',OneHotEncoder(drop='first', handle_unknown='ignore'),['job','marital','housing','loan','contact','month','day_of_week','poutcome']),
    ('sts',StandardScaler(),['age','duration','campaign','pdays','previous','emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed','previously_contacted'])
    ],remainder='passthrough'
)


X_train_transformed = transform.fit_transform(X_train)
X_test_transformed = transform.transform(X_test)

In [ ]:
clf = RandomForestClassifier()

param_grid ={

    "n_estimators":[300,400,500],
    "max_depth":[12,13,14],
    "min_samples_split":[7,9,11],
    "min_samples_leaf":[4,5,7],
    "criterion":["gini","entropy"],
    "class_weight":["balanced",None],
    "max_samples":[.5,.7,.9]
}

random_search = RandomizedSearchCV(estimator=clf, param_distributions=param_grid,n_iter=10,cv=5,scoring="roc_auc",n_jobs=-1,random_state=42)

random_search.fit(X_train_transformed,Y_train)

print("Best Parameters:", random_search.best_params_)
print("Best ROC-AUC Score:", random_search.best_score_)


Best Parameters: {'n_estimators': 400, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_samples': 0.7, 'max_depth': 14, 'criterion': 'entropy', 'class_weight': None}
Best ROC-AUC Score: 0.9460122147122416


In [ ]:
best_rf = random_search.best_estimator_
y_probs_rf = best_rf.predict_proba(X_test_transformed)[:, 1]
y_pred_rf = best_rf.predict(X_test_transformed)

print(
    f"Test ROC-AUC: {roc_auc_score(Y_test, y_probs_rf):.4f}"
)  # Expect ~0.944+

Test ROC-AUC: 0.9467


In [ ]:
print(classification_report(Y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.93      0.98      0.95      6871
           1       0.71      0.36      0.48       831

    accuracy                           0.92      7702
   macro avg       0.82      0.67      0.72      7702
weighted avg       0.90      0.92      0.90      7702



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

clf_gb = GradientBoostingClassifier()

param_grid_gb = {
    "n_estimators":[400,500],
    "max_depth":[13,14],
    "min_samples_split":[9,11,13],
    "min_samples_leaf":[6,7,8],
    "max_leaf_nodes":[14,16,20],
    "criterion":["friedman_mse","squared_error"],
    "subsample":[.6,.7,.8],

}

random_search_gb = RandomizedSearchCV(estimator=clf_gb, param_distributions=param_grid_gb,n_iter=10,cv=5,scoring="roc_auc",n_jobs=-1,random_state=42)

random_search_gb.fit(X_train_transformed,Y_train)

print("Best Parameters:", random_search_gb.best_params_)
print("Best ROC-AUC Score:", random_search_gb.best_score_)

Best Parameters: {'subsample': 0.8, 'n_estimators': 500, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_leaf_nodes': 16, 'max_depth': 14, 'criterion': 'squared_error'}
Best ROC-AUC Score: 0.9468427543364271


In [ ]:
best_gb = random_search_gb.best_estimator_
y_probs_gb = best_gb.predict_proba(X_test_transformed)[:, 1]
y_pred_gb = best_gb.predict(X_test_transformed)

print(
    f"Test ROC-AUC: {roc_auc_score(Y_test, y_probs_gb):.4f}"
)  # Expect ~0.9468+
print(classification_report(Y_test, y_pred_gb))

Test ROC-AUC: 0.9471
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      6871
           1       0.63      0.51      0.57       831

    accuracy                           0.92      7702
   macro avg       0.79      0.74      0.76      7702
weighted avg       0.91      0.92      0.91      7702



In [16]:
xgb_clf = XGBClassifier(random_state=100, eval_metric='logloss')


param_grid_xgb = {
    "n_estimators": [400, 500, 600],
    "max_depth": [6, 8, 9],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.4, 0.5, 0.6],
    "colsample_bytree": [0.4, 0.5, 0.6, 0.7],
    "min_child_weight": [0.0, 0.01, 0.05],
    "reg_alpha": [0.0, 0.01, 0.1],
    "reg_lambda": [0.0, 0.01, 0.1],
    "scale_pos_weight": [1, 2, 3]
}

random_search_xgb = RandomizedSearchCV(estimator=xgb_clf,param_distributions=param_grid_xgb,n_iter=10,cv=10,scoring="roc_auc",n_jobs=-1,random_state=42)

random_search_xgb.fit(X_train_transformed, Y_train)

best_xgb = random_search_xgb.best_estimator_
print("Best Parameters:", random_search_xgb.best_params_)

y_probs_xgb = best_xgb.predict_proba(X_test_transformed)[:, 1]
print(f"XGBoost Test ROC-AUC: {roc_auc_score(Y_test, y_probs_xgb):.4f}")

Best Parameters: {'subsample': 0.6, 'scale_pos_weight': 2, 'reg_lambda': 0.0, 'reg_alpha': 0.0, 'n_estimators': 400, 'min_child_weight': 0.01, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
XGBoost Test ROC-AUC: 0.9475


In [17]:
y_pred_xgb = best_xgb.predict(X_test_transformed)

print(classification_report(Y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      6871
           1       0.59      0.71      0.64       831

    accuracy                           0.91      7702
   macro avg       0.78      0.83      0.80      7702
weighted avg       0.92      0.91      0.92      7702



In [16]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 11.9 MB/s eta 0:00:00


In [17]:
import optuna

from sklearn.model_selection import cross_val_score

In [38]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    max_depth = trial.suggest_int('max_depth', 1, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.03)
    subsample = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)
    min_child_weight = trial.suggest_float('min_child_weight', 0.0, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 0.0, 1.0)
    reg_lambda = trial.suggest_float('reg_lambda', 0.0, 1.0)
    scale_pos_weight = trial.suggest_int('scale_pos_weight', 1, 10)

    Model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    )

    scores = cross_val_score(Model, X_train_transformed, Y_train, cv=5, scoring='f1')
    return scores.mean()

In [ ]:
study = optuna.create_study(direction='maximize',sampler = optuna.samplers.TPESampler())
study.optimize(objective, n_trials=100)

[I 2026-10-15 13:05:54,275] A new study created in memory with name: no-name-0a366220-8986-4758-88ee-d28fa38ac414
[I 2026-10-15 13:05:59,597] Trial 0 finished with value: 0.6299737599834213 and parameters: {'n_estimators': 101, 'max_depth': 9, 'learning_rate': 0.017494439697611212, 'subsample': 0.8823352108970528, 'colsample_bytree': 0.8474530528985824, 'min_child_weight': 0.6527928051327796, 'reg_alpha': 0.9549436813157337, 'reg_lambda': 0.5854960291385521, 'scale_pos_weight': 2}. Best is trial 0 with value: 0.6299737599834213.
[I 2026-10-15 13:06:26,332] Trial 1 finished with value: 0.6245413200050477 and parameters: {'n_estimators': 877, 'max_depth': 5, 'learning_rate': 0.012861135436513384, 'subsample': 0.5190218713712875, 'colsample_bytree': 0.8923796752364181, 'min_child_weight': 0.6673382442557649, 'reg_alpha': 0.7243066356601868, 'reg_lambda': 0.9868775042425582, 'scale_pos_weight': 7}. Best is trial 0 with value: 0.6299737599834213.
[I 2026-10-15 13:06:39,627] Trial 2 finished

In [40]:
op_best_params = study.best_trial.params


print(f'Best roc_auc: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best roc_auc: 0.6559711483040007
Best hyperparameters: {'n_estimators': 642, 'max_depth': 5, 'learning_rate': 0.0250358271650886, 'subsample': 0.7229431832053889, 'colsample_bytree': 0.9534417835812732, 'min_child_weight': 0.7544241266131934, 'reg_alpha': 0.39676225131536896, 'reg_lambda': 0.9597434684998057, 'scale_pos_weight': 3}


In [20]:
best_opt_model = XGBClassifier(n_estimators = 642, max_depth = 5, learning_rate = 0.0250358271650886, subsample = 0.7229431832053889, colsample_bytree = 0.9534417835812732, min_child_weight = 0.7544241266131934, reg_alpha = 0.39676225131536896, reg_lambda = 0.9597434684998057, scale_pos_weight = 3, random_state=42, eval_metric='logloss')
best_opt_model.fit(X_train_transformed, Y_train)


op_pred = best_opt_model.predict(X_test_transformed)
print(classification_report(Y_test, op_pred))

              precision    recall  f1-score   support

           0       0.97      0.92      0.94      6871
           1       0.54      0.79      0.64       831

    accuracy                           0.90      7702
   macro avg       0.76      0.85      0.79      7702
weighted avg       0.93      0.90      0.91      7702



In [21]:
y_probs_opt = best_opt_model.predict_proba(X_test_transformed)[:, 1]
print(f"Optuna-tuned XGBoost Test ROC-AUC: {roc_auc_score(Y_test, y_probs_opt):.4f}")

Optuna-tuned XGBoost Test ROC-AUC: 0.9482


In [22]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [42]:
plot_parallel_coordinate(study).show()

In [43]:
plot_param_importances(study).show()


In [23]:
model_pipeline = make_pipeline(
    transform,
    best_opt_model
)

model_pipeline.fit(X_train,Y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ore',
                                                  OrdinalEncoder(categories=[['illiterate',
                                                                              'basic.4y',
                                                                              'basic.6y',
                                                                              'basic.9y',
                                                                              'high.school',
                                                                              'professional.course',
                                                                              'university.degree']]),
                                                  ['education']),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['job', 'marital', 'housing',
                                                   'loan', 'contact'...
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.0250358271650886, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=5,
                               max_leaves=None,
                               min_child_weight=0.7544241266131934, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=642, n_jobs=None,
                               num_parallel_tree=None, ...))])